In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
from point_cloud import load_rgbd_image, rgbd_to_points, load_rgbd_image_pair, rgbd_to_points_orthographic
import open3d as o3d
import numpy as np
from depth.depth_anything_model import DepthAnythingModel
import matplotlib.pyplot as plt
import cv2
from image import resize_image_max_px

In [ ]:
filename = "image_00000.png"
directory = "output"
max_px = 1024

color_image, depth_image = load_rgbd_image_pair(filename, directory)
color_image = resize_image_max_px(color_image, max_px)

In [ ]:
rgbd_image = load_rgbd_image(filename, directory, 1.0)
pcd = rgbd_to_points(rgbd_image, 1.0, 1, 1)

cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
inlier_cloud = pcd.select_by_index(ind)
o3d.visualization.draw_geometries([inlier_cloud])
print("Done!")

In [ ]:
model = DepthAnythingModel("depth_anything_vitl14")

In [ ]:
image_path = "path/to/image.jpg"
color_image = cv2.imread(image_path)
color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
color_image = resize_image_max_px(color_image, 2500)

In [ ]:
depth = model.predict_depth(color_image, False)
# min_distance = 300
min_distance = None
max_depth = depth.max()
if min_distance is None:
    depth = (max_depth - depth) + (max_depth * 1.0) # why 0.5? Idk
else:
    depth = (max_depth - depth) + min_distance

plt.imshow(depth)
plt.show()

In [ ]:
depth_scale = 3

height, width, _ = depth.shape
fx, fy = height * 0.6, width * 0.6
x, y = np.meshgrid(np.arange(width), np.arange(height))
x = (x - width / 2) / fx
y = (y - height / 2) / fy

z = depth.squeeze()

# z = np.random.rand(*z.shape)
# points = np.stack((x, y, z), axis=-1).reshape(-1, 3)
points = np.stack((np.multiply(x, -z), np.multiply(y, -z), z), axis=-1).reshape(-1, 3)
points[:, 2] *= depth_scale
colors = color_image.reshape(-1, 3)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)

# pcd.transform([[-1, 0, 0, 0], [0, 1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])

cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
inlier_cloud = pcd.select_by_index(ind)

In [ ]:
o3d.visualization.draw_geometries([inlier_cloud])

In [ ]:
orthographic = rgbd_to_points_orthographic(color_image, depth, 1.0)